# 02 — Activations and Metadata

**Purpose:** Walk through every field on an `Op` record that carries payload, shape,
dtype, device, memory, timing, compute, args, RNG state, and source-code context.
Shows `Quantity` formatting (`Bytes`, `Duration`, `Flops`, `Macs`) via f-strings.

**Surfaces covered:**
- [ ] `op.out` — captured output tensor
- [ ] `op.shape` — output shape tuple
- [ ] `op.dtype` — torch dtype
- [ ] `op.device_ref` — `DeviceRef` (`.backend`, `.name`)
- [ ] `op.activation_memory` — raw byte count
- [ ] `Bytes` / `Duration` / `Flops` / `Macs` — `Quantity` classes + f-string formatting
- [ ] `op.func_duration` — wall-clock time for the op
- [ ] `op.flops_forward` / `op.macs_forward` — compute counts
- [ ] `op.arg_expressions` — symbolic arg names
- [ ] `op.saved_args` — captured input tensors (requires `save_args=True`)
- [ ] `op.arg_names` — parameter-style names (e.g. `('input', 'weight', 'bias')`)
- [ ] `op.func_rng_states` — RNG state dict (requires `save_rng=True`)
- [ ] `op.code_context` — list of `FuncCallLocation` records
- [ ] `FuncCallLocation` — `.call_line`, `.code_context`, repr- [ ] `tl.trace(..., save=SaveOptions(activation_transform=), capture=CaptureOptions(seed=))` — capture-shaping options


## 1. Environment setup

In [ ]:
import pathlib
import sys

# Pin imports to THIS checkout: the notebook dir (for _models.py) plus the repo
# root, so `import torchlens` audits the code this notebook ships with -- not a
# pip-installed copy from another checkout.
_NB_DIR = pathlib.Path.cwd()
if not (_NB_DIR / "_models.py").exists():
    _NB_DIR = next(
        p
        for p in [_NB_DIR / "notebooks" / "audit", *_NB_DIR.parents]
        if (p / "_models.py").exists()
    )
_REPO_ROOT = _NB_DIR.parents[1]
sys.path.insert(0, str(_NB_DIR))
sys.path.insert(0, str(_REPO_ROOT))

import warnings as _w

_w.filterwarnings("ignore", message=".*pynvml.*")  # torch/cuda probe noise
import torch

import torchlens as tl

assert pathlib.Path(tl.__file__).is_relative_to(_REPO_ROOT), (
    f"torchlens imported from {tl.__file__} -- expected this checkout"
)
from _models import ZOO

from torchlens.types import Bytes, Duration, Flops, Macs, Quantity

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. Capture a trace — baseline model

Use `tiny_mlp` for simple fields; `demo_model` later for RNG state.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

print(repr(trace))
print("layer_labels:", trace.layer_labels)

# Grab the linear and relu ops for metadata inspection
op_lin = trace["linear_1_1"]  # linear op
op_relu = trace["relu_1_2"]  # relu op

## 3. `op.out` — output tensor

The captured output tensor.  Returned as a detached copy by default so it is safe to
inspect and mutate independently of the computation graph.

In [ ]:
print("op_relu.out type :", type(op_relu.out).__name__)
print("op_relu.out      :")
print(op_relu.out)
print()
print("Is grad retained in .out?", op_relu.out.grad_fn)

## 4. `op.shape`, `op.dtype`, `op.device_ref`

In [ ]:
print("op_lin.shape      :", op_lin.shape)
print("op_lin.dtype      :", op_lin.dtype)
print()
dev = op_lin.device_ref
print("op_lin.device_ref :", dev)
print("  type            :", type(dev).__name__)
print("  .backend        :", dev.backend)
print("  .name           :", dev.name)

## 5. `op.activation_memory` and `Bytes` formatting

`activation_memory` is the raw byte count of the output tensor.  
`Bytes` is a `Quantity` subclass with a human-readable `__str__` / `__format__`.

In [ ]:
raw_bytes = op_lin.activation_memory
print("op_lin.activation_memory (raw int):", raw_bytes)

b = Bytes(raw_bytes)
print("Bytes(raw) str    :", b)  # auto format
print("Bytes(raw) repr   :", repr(b))  # repr also unit-formatted (fixed 2026-06)
print(f"f-string          : {b}")  # same via __format__
print()

# Show scaling at different sizes
for n_bytes in [512, 2**10, 2**20, 2**30]:
    print(f"  {n_bytes:>12,} B  ->  {Bytes(n_bytes)}")

## 6. `Duration` — wall-clock timing per op

`op.func_duration` holds seconds as a float; wrapping in `Duration` humanises it.

In [ ]:
raw_dur = op_lin.func_duration
print("op_lin.func_duration (raw float, seconds):", raw_dur)

d = Duration(raw_dur)
print("Duration(raw)  :", d)
print(f"f-string       : {d}")
print()

# Duration across every op in the trace
print("Per-op timing:")
for lbl in trace.layer_labels:
    op = trace[lbl]
    print(f"  {lbl:<18}: {Duration(op.func_duration)}")

## 7. `Flops` and `Macs` — compute counts

`op.flops_forward` counts FLOPs (convention: one multiply-accumulate = 2 FLOPs).  
`op.macs_forward` is the MAC count (= FLOPs // 2).  
Both are 0 for non-compute ops (input, output).

In [ ]:
raw_flops = op_lin.flops_forward
raw_macs = op_lin.macs_forward
print("op_lin.flops_forward (raw):", raw_flops)
print("op_lin.macs_forward  (raw):", raw_macs)
print()

fl = Flops(raw_flops)
mc = Macs(raw_macs)
print("Flops(raw) :", fl)
print(f"f-string   : {fl}")
print("Macs(raw)  :", mc)
print(f"f-string   : {mc}")
print()

# Compute summary across all ops
print("Per-op compute:")
for lbl in trace.layer_labels:
    op = trace[lbl]
    flops = Flops(op.flops_forward) if op.flops_forward is not None else "n/a"
    macs = Macs(op.macs_forward) if op.macs_forward is not None else "n/a"
    print(f"  {lbl:<18}: FLOPs={flops}  MACs={macs}")
    # macs_forward is None (not zero) on ops with no MAC convention --
    # the A07 honesty re-basing: absent is absent, never a fabricated 0.

## 8. `op.arg_names` and `op.arg_expressions`

`arg_names` are the formal parameter names from the torch function signature  
(e.g. `('input', 'weight', 'bias')` for `F.linear`).  
`arg_expressions` are symbolic expressions describing what was passed  
(e.g. `['input', 'self.weight', 'self.bias']`).  

They are populated during default tracing.  `saved_args` (the actual tensors) requires
`save_args=True` (see section 9).

In [ ]:
print("=== linear op ===")
print("  arg_names       :", op_lin.arg_names)
print("  arg_expressions :", op_lin.arg_expressions)
print("  has_saved_args  :", op_lin.has_saved_args)
print()

print("=== relu op ===")
print("  arg_names       :", op_relu.arg_names)
print("  arg_expressions :", op_relu.arg_expressions)

## 9. `op.saved_args` — captured input tensors (`save_args=True`)

By default `saved_args` is `None` to avoid storing every intermediate tensor twice.
Pass `save_args=True` to `tl.trace` to retain them.

In [ ]:
model2, x2 = ZOO["tiny_mlp"]()
trace_args = tl.trace(model2, x2, capture=tl.options.CaptureOptions(save_args=True))

op_lin2 = trace_args["linear_1_1"]
print("op_lin2.has_saved_args :", op_lin2.has_saved_args)
print("op_lin2.saved_args     :", op_lin2.saved_args)
print()

if op_lin2.saved_args is not None:
    for i, t in enumerate(op_lin2.saved_args):
        print(f"  arg[{i}] shape={t.shape} dtype={t.dtype}")

## 10. `op.func_rng_states` — RNG state at call time (`save_rng=True`)

Captured only for ops that consume randomness (e.g. `torch.rand`, `dropout`).  
Useful for exact replay.  Requires `save_rng=True`.

In [ ]:
# demo_model has torch.rand in its forward pass
model_d, x_d = ZOO["demo_model"]()
trace_rng = tl.trace(model_d, x_d, capture=tl.options.CaptureOptions(save_rng=True))

rand_labels = [lbl for lbl in trace_rng.layer_labels if "rand" in lbl]
print("rand-type labels:", rand_labels)

if rand_labels:
    op_rand = trace_rng[rand_labels[0]]
    print()
    print("op_rand.func_name      :", op_rand.func_name)
    rng = op_rand.func_rng_states
    print("op_rand.func_rng_states type :", type(rng).__name__)
    print("  keys  :", list(rng.keys()))
    for k, v in rng.items():
        if hasattr(v, "shape"):
            print(f"  {k}: tensor shape={v.shape} dtype={v.dtype}")
        else:
            print(f"  {k}: {type(v).__name__} len={len(v) if hasattr(v, '__len__') else '?'}")
else:
    print("No rand op found in demo_model trace -- check model forward()")

## 11. `op.code_context` and `FuncCallLocation`

`code_context` is a list of `FuncCallLocation` records tracing the Python call stack
at the point where the op was invoked.  By default only the innermost frame is captured.
Pass `save_source_context=True` for the full stack.  Each `FuncCallLocation` has:
- `.call_line` — 1-indexed line number in source
- `.code_context` — the source line text (or `None` if unavailable for dynamic code)
- repr shows file:line:function

In [ ]:
import torch.nn as nn


# Use a named module so source is available
class ExplicitModel(nn.Module):
    """Tiny model defined in-notebook so code_context has a real source line."""

    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(self.fc(x))  # <-- this line captured in code_context


model_e = ExplicitModel()
x_e = torch.randn(2, 4)
trace_ctx = tl.trace(model_e, x_e, capture=tl.options.CaptureOptions(save_source_context=True))

op_ctx = trace_ctx["relu"]
cc = op_ctx.code_context

print(f"code_context: {len(cc)} frame(s)")
for i, fl in enumerate(cc):
    print(f"\n--- frame {i} ---")
    print("  repr        :", repr(fl))
    print("  .call_line  :", fl.call_line)
    print("  .code_context:", fl.code_context)

## 12. Full metadata snapshot on one op

Collect all the above fields into a single printout so a reviewer can see what a user
encounters when they call `print(vars(op))` or iterate fields.

In [ ]:
model3, x3 = ZOO["tiny_mlp"]()
trace_full = tl.trace(
    model3,
    x3,
    capture=tl.options.CaptureOptions(save_args=True, save_source_context=True),
)
op_full = trace_full["linear_1_1"]

print("=== Full metadata snapshot: linear_1_1 ===")
print(f"  func_name          : {op_full.func_name}")
print(f"  layer_label        : {op_full.layer_label}")
print(f"  raw_label          : {op_full.raw_label}")
print(f"  shape              : {op_full.shape}")
print(f"  dtype              : {op_full.dtype}")
print(f"  device_ref.name    : {op_full.device_ref.name}")
print(f"  activation_memory  : {Bytes(op_full.activation_memory)}")
print(f"  func_duration      : {Duration(op_full.func_duration)}")
print(f"  flops_forward      : {Flops(op_full.flops_forward)}")
print(f"  macs_forward       : {Macs(op_full.macs_forward)}")
print(f"  arg_names          : {op_full.arg_names}")
print(f"  arg_expressions    : {op_full.arg_expressions}")
print(f"  has_saved_args     : {op_full.has_saved_args}")
saved = op_full.saved_args
if saved:
    print(f"  saved_args shapes  : {[tuple(t.shape) for t in saved]}")
cc = op_full.code_context
if cc:
    print(f"  code_context[0]    : {repr(cc[0])}")

## 13. Other capture-time options — `activation_transform`, `save=` predicate, `seed`

Three `tl.trace` kwargs that shape what gets captured: `activation_transform` maps
every saved payload through a callable (e.g. downcast to fp16 to halve memory),
a `save=` predicate (a selector such as `tl.in_module(...)`) restricts which sites keep payloads, and `seed` pins RNG for
reproducible traces.

In [ ]:
model, x = ZOO["tiny_mlp"]()

# activation_transform: SHOULD downcast every saved activation to fp16.
# Assert the effect rather than just printing it -- inert kwargs must not demo as working.
t_half = tl.trace(
    model, x, capture=tl.options.CaptureOptions(activation_transform=lambda t: t.half())
)
dt = t_half["relu_1_2"].out.dtype
print("activation_transform=half: relu payload dtype:", dt)
if dt != torch.float16:
    print("⚠️ GAP: the transform did NOT apply -- payload is still", dt)
print()

# save= predicate: restricts which sites keep payloads. Selectors compose with & | ~, so
# "this module AND this function" is one expression -- there is no separate module filter.
t_filt = tl.trace(model, x, save=tl.in_module("in_proj"))
saved = [lb for lb in t_filt.layer_labels if getattr(t_filt[lb], "has_saved_activation", False)]
print("save=tl.in_module('in_proj'): ops with saved payloads:", saved)
print()

# random_seed: two traces with the same seed produce identical dropout masks
import torch.nn as nn


class DropModel(nn.Module):
    def forward(self, x):
        return torch.dropout(x, p=0.5, train=True)


dm, dx = DropModel(), torch.randn(2, 8)
t_a = tl.trace(dm, dx, capture=tl.options.CaptureOptions(seed=123))
t_b = tl.trace(dm, dx, capture=tl.options.CaptureOptions(seed=123))
print("seed=123 twice -> identical dropout output:", torch.equal(t_a[-1].out, t_b[-1].out))

---

## ⚠️ GAPs / ergonomic smells

- *(fixed since 2026-06)* **`Quantity.__repr__` now carries units** — `repr(Bytes(2048))`
  is `Bytes('2.0 KB')`, `repr(Duration(0.0123))` is `Duration('12.3 ms')`. The old
  bare-number repr GAP is closed.

- **`op.arg_expressions` population is inconsistent across ops** in a default trace:
  some ops (e.g. `linear`) populate symbolic expressions without any flag, others are
  empty. The exact condition (which ops, which flags) is not documented.

- **`FuncCallLocation.code_context` is `None` for dynamically-defined models**
  (lambda/`<string>` source). The repr shows `code: source unavailable` which is clear,
  but the attribute returning `None` (vs a sentinel string) makes formatting awkward —
  callers must None-check.

- **`activation_transform=` is INERT** — `lambda t: t.half()` leaves saved payloads
  float32 (§13 asserts this). Sibling `grad_transform=` is equally inert (notebook 06).
- **`FuncCallLocation.__repr__` embeds OSC-8 terminal hyperlink escape sequences**
  (`\x1b]8;;file://...`) — clickable in a modern terminal, garbage in notebooks, logs,
  and diffs. Reprs should be plain text; hyperlinks belong in an opt-in rich display.

- **`op.func_rng_states` is `{}` unless `save_rng=True`**, with no hint in
  `op.__repr__` that RNG states exist and could be captured. A note like
  `rng_states: not captured (pass save_rng=True)` would help discoverability.